# KPI Definition & Business Metric Design

This notebook demonstrates our organization-wide KPI measurement and validation framework. We import the reusable python module from `/kpis/kpi_functions.py`, calculate the 5 core KPIs, validate them against targets from `kpi_validation_targets.json`, and perform hierarchical revenue decomposition.

## Setup & Loading Data

In [ ]:
import os
import sys
import json
import pandas as pd

# Ensure we can import from project root and kpis directory
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../kpis"))

from kpis.kpi_functions import (
    load_data,
    calculate_mau,
    calculate_revenue_per_customer,
    calculate_churn_rate,
    calculate_payment_success_rate,
    calculate_customer_acquisition_cost,
    validate_kpis,
    decompose_revenue
)

RAW_DATA_PATH = "../data/raw/kpi_transactions.csv"
df = load_data(RAW_DATA_PATH)
df.head()

## Task 1 & 2: Compute Core KPIs

We calculate the 5 Key Performance Indicators defined in `/kpis/kpi_reference.md`:

In [ ]:
ref_date = df['transaction_date'].max()

mau = calculate_mau(df, reference_date=ref_date)
rpc = calculate_revenue_per_customer(df)
churn = calculate_churn_rate(df, reference_date=ref_date)
pay_rate = calculate_payment_success_rate(df)
cac = calculate_customer_acquisition_cost(df, total_spend=120000.0, reference_date=ref_date)

current_kpis = {
    'mau': mau,
    'revenue_per_customer': rpc,
    'churn_rate': churn,
    'payment_success_rate': pay_rate,
    'customer_acquisition_cost': cac
}

print("=== COMPUTED ACTUAL KPI VALUES ===")
print(f"Monthly Active Users (MAU): {mau:,}")
print(f"Revenue per Customer (RPC): ${rpc:.2f}")
print(f"Churn Rate:                 {churn:.2%}")
print(f"Payment Success Rate:       {pay_rate:.2%}")
print(f"Customer Acquisition Cost:  ${cac:.2f}")

## Task 3: Validate Against Target Thresholds

We load target configurations from `../kpis/kpi_validation_targets.json` and validate each computed KPI. Thresholds are set to alert stakeholders when metrics fall outside desired ranges.

In [ ]:
targets_path = "../kpis/kpi_validation_targets.json"
validation_df = validate_kpis(current_kpis, targets_path=targets_path)

print("\n=== VALIDATION REPORT STATUS ===")
validation_df[['kpi_name', 'formatted_actual', 'target_range', 'status']]

## Task 4: Revenue KPI Decomposition

Top-level metrics can hide underlying details. To isolate drivers, we decompose cumulative revenue hierarchically:
- **Level 1**: Top-level Revenue sum.
- **Level 2**: Revenue split by Customer Segment (Enterprise vs. SMB vs. Startup).
- **Level 3**: Product split within each segment.

In [ ]:
decompose_revenue(df)

## Strategic Discussion

### Difference Between a Metric and a KPI
- A **Metric** is a raw quantitative value representing business activity (e.g. "10,000 transaction events"). It is absolute and does not signify performance context.
- A **KPI** is a ratio or rate (e.g. "Payment Success Rate = 97.9%") measured against a target range. It indicates strategic health and provides a clear signal on whether an intervention is required.

### Target Setting
- Targets are established using historical performance baselines combined with stretch expectations. For example, a 95% target for payment success rate is an operational SLA. Churn rate limits are modeled on industry benchmarks to preserve compounding subscriber growth.

### Managing Schema Changes
- When schemas evolve, the KPI computation functions in `/kpis/kpi_functions.py` abstract the database details. By keeping definitions centralized in a single module, teams can modify column mappings in one place without breaking reports, notebooks, or automated dashboards.